# A6 — Clasificación retórica con Gemini vía API

Evaluación de Gemini 2.5 Flash para segmentación retórica de documentos científicos en español.

Se comparan dos estrategias sobre el conjunto EVAL (1699 fragmentos, 8 etiquetas):
- **Zero-shot** (temperatura 0, llamada única): baseline sin contexto previo.
- **Few-shot + majority voting** (k=3, temperatura 0.5): 8 ejemplos en el prompt, 3 votos por fragmento, gana la etiqueta más votada; empates se resuelven por confianza acumulada.

Los ejemplos few-shot se cargan del mismo archivo `fewshot_examples.json` que usa el API desplegado, garantizando que el notebook evalúa exactamente lo que corre en producción.

Dataset: `Dataset_consolidado_final_v4.csv` — partición `dataset_type == EVAL`.  
SDK: `google-genai`.

In [1]:
!pip install -q google-genai pandas scikit-learn matplotlib seaborn tqdm

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import asyncio
import json
import os
import re
import time
from collections import Counter
from pathlib import Path

from google import genai
from google.genai import types
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from tqdm.auto import tqdm

DRIVE_ROOT  = Path("/content/drive/MyDrive")
EVAL_PATH   = DRIVE_ROOT / "Dataset_consolidado_final_v4.csv"
FEWSHOT_PATH = DRIVE_ROOT / "fewshot_examples.json"   # mismo archivo del API
RESULTS_DIR = DRIVE_ROOT / "results_a6_v4"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "gemini-2.5-flash"

COST_INPUT_PER_M  = 0.15
COST_OUTPUT_PER_M = 0.60

MAX_TEXT_WORDS = 700
CONCURRENCY    = 10
LABELS         = ["INTRO", "BACK", "METH", "RES", "DISC", "CONTR", "LIM", "CONC"]

ZEROSHOT_TEMP = 0

VOTING_K    = 3     # llamadas por fragmento
VOTING_TEMP = 0.5   # temperatura > 0 para variabilidad entre votos

api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
if not api_key:
    raise EnvironmentError("GOOGLE_API_KEY no encontrada. Agregar en Colab Secrets.")

client = genai.Client(api_key=api_key)
print(f"Modelo:              {MODEL}")
print(f"EVAL path:           {EVAL_PATH}")
print(f"Fewshot examples:    {FEWSHOT_PATH}")
print(f"Results dir:         {RESULTS_DIR}")
print(f"Zero-shot temp:      {ZEROSHOT_TEMP}")
print(f"Few-shot k={VOTING_K}, temp={VOTING_TEMP}")

In [6]:
df_all   = pd.read_csv(EVAL_PATH)
df_eval  = df_all[df_all["dataset_type"] == "EVAL"].reset_index(drop=True)
df_train = df_all[df_all["dataset_type"].isin(["TRAIN", "TEST"])].reset_index(drop=True)

print("Distribución EVAL:")
print(df_eval["label"].value_counts().to_string())
print(f"\nTotal EVAL:       {len(df_eval)}")
print(f"Total TRAIN+TEST: {len(df_train)}")

Distribución EVAL:
label
INTRO    259
DISC     229
BACK     226
RES      207
METH     202
CONC     200
CONTR    192
LIM      184

Total EVAL:       1699
Total TRAIN+TEST: 15334


In [7]:
SYSTEM_PROMPT = """Eres un experto en análisis del discurso científico en español.
Clasifica el fragmento textual de un artículo científico en una de estas 8 categorías retóricas.

DEFINICIONES Y SEÑALES PRIMARIAS:

CONTR — El fragmento DECLARA EXPLÍCITAMENTE el aporte original del trabajo.
  Señales: verbos en primera persona del plural referidos al propio trabajo ("proponemos",
  "presentamos", "desarrollamos", "describimos"), frases como "nuestra contribución es",
  "a diferencia de trabajos previos, este trabajo/método/sistema", "el aporte principal
  de este artículo es", "este trabajo introduce/propone/presenta un nuevo".
  NO es INTRO (que plantea objetivos sin declarar aportes) ni DISC (que interpreta resultados).
  REGLA: ante cualquier señal de declaración de aporte original, clasifica como CONTR.

BACK — Describe trabajos PREVIOS de OTROS autores. No habla del estudio actual.
  Señal principal: citas bibliográficas en cualquier formato ([1], (Autor, año), [Autor et al.],
  "según X", "Y et al. demostraron", "estudios previos de Z").
  Si el fragmento menciona trabajos ajenos o incluye citas, es BACK aunque parezca INTRO.
  NO es INTRO (sin citas, habla del problema actual) ni DISC (interpreta resultados propios).

METH — Explica qué se hizo: diseño experimental, métodos, datos, materiales, procedimientos.
  Describe los pasos seguidos para realizar el estudio.
  NO es LIM: si el énfasis está en restricciones o fallas del método, es LIM, no METH.

RES — Presenta solo los resultados obtenidos: números, porcentajes, tablas, comparaciones empíricas.
  Sin interpretación de qué significan esos datos.
  NO es DISC: si el fragmento dice "esto sugiere", "esto indica", "esto demuestra", es DISC.
  NO es BACK: si los resultados son del propio estudio (no cita trabajos ajenos), es RES.

DISC — Interpreta los resultados del estudio actual, analiza implicaciones, compara con hipótesis.
  Frases como "estos resultados sugieren", "esto indica que", "en comparación con [hipótesis]".
  NO es RES (que solo reporta datos sin interpretar).

LIM — Describe restricciones, supuestos, fuentes de error o limitaciones de generalización.
  Puede mencionar trabajo futuro. El énfasis está en qué NO funciona o qué es imperfecto.
  Frases como "una limitación de este estudio es", "no consideramos", "queda pendiente".
  NO es METH (que describe el método sin señalar sus restricciones).

CONC — Resume los hallazgos principales y presenta las conclusiones finales.
  Suele aparecer al final del artículo. Frases como "en conclusión", "este trabajo demostró".
  NO es DISC (que interpreta resultados) ni LIM (que solo menciona limitaciones).

INTRO — Presenta el problema de investigación, motivación y objetivos del trabajo.
  NO cita trabajos ajenos (eso es BACK). NO declara aportes propios (eso es CONTR).
  Usa INTRO solo si el fragmento plantea el problema o los objetivos sin señales de BACK ni CONTR.

ORDEN DE PRIORIDAD (aplica en orden ante la duda):
1. ¿Declara explícitamente un aporte original? → CONTR
2. ¿Cita trabajos de otros autores? → BACK
3. ¿Resume hallazgos finales o concluye el artículo? → CONC
4. ¿Presenta solo datos sin interpretar? → RES
5. ¿Describe métodos sin mencionar restricciones? → METH
6. ¿Interpreta qué significan los resultados? → DISC
7. ¿Menciona restricciones o limitaciones? → LIM
8. Ninguno de los anteriores → INTRO

Responde ÚNICAMENTE con un JSON válido en el formato exacto:
{"label": "<UNA DE LAS 8 ETIQUETAS>", "confidence": <número entre 0.0 y 1.0>}

No incluyas explicaciones, markdown ni texto fuera del JSON."""

print("Prompt listo. Configuraciones de generación se definen en la celda de pipeline.")


thinking_budget:   0
max_output_tokens: 150
temperature:       0.5  (variabilidad para voting)
Config lista.


In [ ]:
if not FEWSHOT_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {FEWSHOT_PATH}\n"
        f"Subir api/fewshot_examples.json a la raíz de Google Drive."
    )

with open(FEWSHOT_PATH, encoding="utf-8") as f:
    fewshot_examples: dict[str, str] = json.load(f)

missing = [l for l in LABELS if l not in fewshot_examples]
if missing:
    raise ValueError(f"Etiquetas faltantes en fewshot JSON: {missing}")

print(f"Cargados desde: {FEWSHOT_PATH}")
for label, text in fewshot_examples.items():
    print(f"  {label}: {len(text.split())} palabras")

def build_user_message(text: str) -> str:
    words = text.split()
    if len(words) > MAX_TEXT_WORDS:
        text = " ".join(words[:MAX_TEXT_WORDS])
    return f"Clasifica el siguiente fragmento de un artículo científico en español:\n\n{text}"

def build_fewshot_contents(text: str) -> list:
    """Construye el historial few-shot para un fragmento a clasificar."""
    contents = []
    for label in LABELS:
        contents.append(types.Content(
            role="user",
            parts=[types.Part(text=build_user_message(fewshot_examples[label]))]
        ))
        contents.append(types.Content(
            role="model",
            parts=[types.Part(text=json.dumps({"label": label, "confidence": 1.0}))]
        ))
    contents.append(types.Content(
        role="user",
        parts=[types.Part(text=build_user_message(text))]
    ))
    return contents

print(f"\nTurns por request few-shot: {len(build_fewshot_contents('test'))}")

In [ ]:
def parse_response(raw: str) -> dict | None:
    if not raw:
        return None
    raw = re.sub(r"```(?:json)?\n?", "", raw).replace("```", "").strip()
    m = re.search(r"\{.*?\}", raw, re.DOTALL)
    if not m:
        return None
    try:
        parsed = json.loads(m.group())
        label = str(parsed.get("label", "")).strip().upper()
        if label not in LABELS:
            return None
        return {"label": label, "confidence": float(parsed.get("confidence") or 0.0)}
    except Exception:
        return None

GEN_CONFIG_ZEROSHOT = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    temperature=ZEROSHOT_TEMP,
    max_output_tokens=150,
    thinking_config=types.ThinkingConfig(thinking_budget=0),
)

GEN_CONFIG_FEWSHOT = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    temperature=VOTING_TEMP,
    max_output_tokens=150,
    thinking_config=types.ThinkingConfig(thinking_budget=0),
)

async def call_once(content, config) -> dict | None:
    for attempt in range(4):
        try:
            response = await client.aio.models.generate_content(
                model=MODEL, contents=content, config=config,
            )
            parsed = parse_response(response.text or "")
            if parsed:
                usage = response.usage_metadata
                parsed["tokens_in"]  = usage.prompt_token_count or 0 if usage else 0
                parsed["tokens_out"] = usage.candidates_token_count or 0 if usage else 0
                return parsed
        except Exception as e:
            err = str(e)
            if "429" in err:
                await asyncio.sleep(60)
            elif "503" in err:
                await asyncio.sleep(30)
            else:
                await asyncio.sleep(2 ** attempt)
    return None

async def classify_zeroshot(semaphore, idx, row):
    async with semaphore:
        result = {
            "_idx": idx, "true_label": row["label"], "doc_id": row["doc_id"],
            "label": None, "confidence": None,
            "tokens_in": 0, "tokens_out": 0, "latency_s": 0.0, "error": None,
        }
        content = build_user_message(row["text"])
        t0 = time.time()
        vote = await call_once(content, GEN_CONFIG_ZEROSHOT)
        result["latency_s"] = time.time() - t0
        if vote:
            result["label"]      = vote["label"]
            result["confidence"] = vote["confidence"]
            result["tokens_in"]  = vote["tokens_in"]
            result["tokens_out"] = vote["tokens_out"]
        else:
            result["error"] = "call failed"
        return result

async def classify_fewshot(semaphore, idx, row):
    async with semaphore:
        result = {
            "_idx": idx, "true_label": row["label"], "doc_id": row["doc_id"],
            "label": None, "confidence": None, "vote_agreement": None,
            "tokens_in": 0, "tokens_out": 0, "latency_s": 0.0, "error": None,
        }
        content = build_fewshot_contents(row["text"])
        t0      = time.time()
        votes   = [v for v in await asyncio.gather(
            *[call_once(content, GEN_CONFIG_FEWSHOT) for _ in range(VOTING_K)]
        ) if v is not None]
        result["latency_s"]  = time.time() - t0
        result["tokens_in"]  = sum(v["tokens_in"]  for v in votes)
        result["tokens_out"] = sum(v["tokens_out"] for v in votes)

        if not votes:
            result["error"] = f"all {VOTING_K} votes failed"
            return result

        label_counts = Counter(v["label"] for v in votes)
        winner_count = max(label_counts.values())
        candidates   = [l for l, c in label_counts.items() if c == winner_count]
        winner_label = max(
            candidates,
            key=lambda l: sum(v["confidence"] for v in votes if v["label"] == l)
        )
        winner_votes = [v for v in votes if v["label"] == winner_label]

        result["label"]          = winner_label
        result["confidence"]     = sum(v["confidence"] for v in winner_votes) / len(winner_votes)
        result["vote_agreement"] = winner_count / len(votes)
        return result

async def run_batch(df, classify_fn, output_path, desc=""):
    output_path = Path(output_path)
    if output_path.exists():
        done     = pd.read_csv(output_path)
        done_ids = set(done["_idx"].tolist())
        print(f"Checkpoint: {len(done_ids)} ya procesados de {len(df)}.")
    else:
        done     = pd.DataFrame()
        done_ids = set()

    pending = [(idx, row) for idx, row in df.iterrows() if idx not in done_ids]
    print(f"Pendientes: {len(pending)}")
    if not pending:
        return done

    semaphore   = asyncio.Semaphore(CONCURRENCY)
    all_results = list(done.to_dict("records")) if not done.empty else []
    tasks       = [classify_fn(semaphore, idx, row) for idx, row in pending]
    completed   = 0; failures = 0; consec = 0

    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=desc):
        r = await coro
        all_results.append(r)
        completed += 1
        if r["label"] is None:
            failures += 1; consec += 1
        else:
            consec = 0
        if consec >= 20:
            pd.DataFrame(all_results).to_csv(output_path, index=False)
            raise RuntimeError(f"20 fallos consecutivos — abortando. Último error: {r['error']}")
        if completed % 50 == 0:
            pd.DataFrame(all_results).to_csv(output_path, index=False)
            print(f"\n  [ckpt {completed}/{len(tasks)}] fallos={failures} ({failures/completed:.0%})")

    combined = pd.DataFrame(all_results)
    combined.to_csv(output_path, index=False)
    print(f"Guardado: {output_path} — {len(combined)} filas, {combined['label'].isna().sum()} fallidos")
    return combined

print("Pipeline listo. Configs: zero-shot (temp=0)  |  few-shot k=3 (temp=0.5)")

## 5. Test de conectividad

In [10]:
async def test_single():
    row  = df_eval.iloc[0]
    resp = await client.aio.models.generate_content(
        model=MODEL,
        contents=build_fewshot_contents(row["text"]),
        config=GEN_CONFIG_FEWSHOT,
    )
    raw    = resp.text or ""
    parsed = parse_response(raw)
    u      = resp.usage_metadata
    print(f"Modelo:          {MODEL}")
    print(f"Respuesta:       {raw}")
    print(f"Parse:           {parsed}")
    print(f"Tokens entrada:  {u.prompt_token_count}")
    print(f"Tokens thinking: {getattr(u, "thoughts_token_count", "None")}")
    print(f"Tokens salida:   {u.candidates_token_count}")
    print()
    if parsed:
        print(f"OK — label={parsed["label"]}, confidence={parsed["confidence"]}")
    else:
        print("FALLO — parse retornó None. Revisar respuesta arriba.")

await test_single()

Modelo:          gemini-2.5-flash
Respuesta:       {"label": "INTRO", "confidence": 1.0}
Parse:           {'label': 'INTRO', 'confidence': 1.0}
Tokens entrada:  3157
Tokens thinking: None
Tokens salida:   14

OK — label=INTRO, confidence=1.0


## 6. Estrategia 1 — Zero-shot

In [ ]:
preds_zs = await run_batch(
    df_eval,
    classify_fn=classify_zeroshot,
    output_path=RESULTS_DIR / "predictions_zeroshot.csv",
    desc="zero-shot",
)
print(f"Total: {len(preds_zs)} | Fallidos: {preds_zs['label'].isna().sum()}")

## 7. Estrategia 2 — Few-shot + majority voting (k=3)

In [ ]:
preds_fs = await run_batch(
    df_eval,
    classify_fn=classify_fewshot,
    output_path=RESULTS_DIR / "predictions_fewshot_k3.csv",
    desc=f"few-shot k={VOTING_K}",
)
print(f"Total: {len(preds_fs)} | Fallidos: {preds_fs['label'].isna().sum()}")

## 8. Resultados por estrategia

In [ ]:
def compute_metrics(preds: pd.DataFrame, strategy_label: str) -> dict:
    valid  = preds.dropna(subset=["label"])
    y_true = valid["true_label"].tolist()
    y_pred = valid["label"].tolist()
    return {
        "Estrategia":  strategy_label,
        "N eval":      len(preds),
        "Válidos":     len(valid),
        "Fallidos":    len(preds) - len(valid),
        "Accuracy":    round(accuracy_score(y_true, y_pred), 4),
        "Macro F1":    round(f1_score(y_true, y_pred, average="macro",  labels=LABELS, zero_division=0), 4),
        "Micro F1":    round(f1_score(y_true, y_pred, average="micro",  labels=LABELS, zero_division=0), 4),
        "Costo USD":   round((preds["tokens_in"].sum()  / 1e6) * COST_INPUT_PER_M +
                             (preds["tokens_out"].sum() / 1e6) * COST_OUTPUT_PER_M, 4),
        "Latencia p50 s": round(preds["latency_s"].median(), 2),
        "Latencia p95 s": round(preds["latency_s"].quantile(0.95), 2),
    }


rows = [
    compute_metrics(preds_zs, "Zero-shot (temp=0, k=1)"),
    compute_metrics(preds_fs, f"Few-shot + majority voting (temp={VOTING_TEMP}, k={VOTING_K})"),
]
comparison = pd.DataFrame(rows).set_index("Estrategia")
print(comparison.to_string())

winner = comparison["Macro F1"].idxmax()
print(f"\n── Ganador por Macro F1: {winner} ({comparison.loc[winner, 'Macro F1']:.4f}) ──")

In [ ]:
def plot_confusion(preds, title, outfile):
    valid  = preds.dropna(subset=["label"])
    y_true = valid["true_label"].tolist()
    y_pred = valid["label"].tolist()
    cm      = confusion_matrix(y_true, y_pred, labels=LABELS)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    acc  = accuracy_score(y_true, y_pred)
    maf1 = f1_score(y_true, y_pred, average="macro", labels=LABELS, zero_division=0)

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues",
                xticklabels=LABELS, yticklabels=LABELS,
                vmin=0, vmax=1, ax=ax, linewidths=0.3)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
    ax.set_title(f"{title}  |  Acc={acc:.3f}  Macro F1={maf1:.3f}")
    plt.tight_layout()
    fig.savefig(outfile, dpi=150, bbox_inches="tight")
    plt.show()
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))


plot_confusion(preds_zs, "Zero-shot", RESULTS_DIR / "cm_zeroshot.png")
plot_confusion(preds_fs, f"Few-shot k={VOTING_K} majority voting", RESULTS_DIR / "cm_fewshot_k3.png")

In [ ]:
comparison.reset_index().to_csv(RESULTS_DIR / "summary_a6.csv", index=False)
print(f"Guardado: {RESULTS_DIR / 'summary_a6.csv'}")
print()
print(comparison[["Accuracy", "Macro F1", "Micro F1", "Costo USD"]].to_string())